# LUMI / Uni-Mol — Jupyter CPU Validation (GitHub-ready v2)

**目的：** 在 HPC 的既有 Jupyter CPU kernel 中，依序完成環境檢查、RDKit、LMDB、公開原始碼取得、LUMI imports、dataset views、CPU forward/backward、3-step mini-training 與 checkpoint 驗證。

**與上一版不同：** 不需要 `git`、`sbatch`、GPU 或任何 pretrained weights；使用 GitHub API 下載 ZIP，來源檔案保存在固定的 `lumi_notebook_workspace/sources/`。已下載的 repo 會重用。**請刪除舊的 git clone 測試 Cells，不要執行 `execute([...])` 的舊段落。**

**使用規範：** 只傳送公開或經機構核准的程式碼，不要放置病人資訊、院內資料、密碼或其他受限制內容。GitHub 可連線不等於機構已批准使用。

## 0. 設定（第一次使用）

首次使用保持 `INSTALL_PACKAGES=True`、`DOWNLOAD_GITHUB=True`。只要你已經完成安裝，下次可以改成 `False`。修改 cell 後從頭順序執行。資料和結果只會寫到這份 Notebook 旁的 `lumi_notebook_workspace/`。

In [ ]:
from pathlib import Path
import os, sys, platform, subprocess, importlib, importlib.util, traceback, json, time, shutil
from IPython.display import display, Markdown

INSTALL_PACKAGES = True
DOWNLOAD_GITHUB = True  # False = reuse sources already downloaded
RUN_MODEL_TESTS = True
MINI_TRAIN_STEPS = 3
SEED = 42

ROOT = Path.cwd().resolve()
WORK = ROOT / "lumi_notebook_workspace"
SOURCE = WORK / "sources"
DATA = WORK / "data"
LOGS = WORK / "logs"
OUTPUT = WORK / "results"
for d in [SOURCE, DATA, LOGS, OUTPUT]: d.mkdir(parents=True,exist_ok=True)
STATUS = {}

def mark(name, status, details=""):
    STATUS[name] = {"test":name,"status":status,"details":str(details)}
    print(f"{'✅' if status=='PASS' else '⚠️' if status=='SKIPPED' else '❌'} {status}: {name}  {details}")

def execute(cmd, label, cwd=ROOT, env=None, timeout=900):
    cmd=list(map(str,cmd))
    print("$", " ".join(cmd))
    try:
        p=subprocess.run(cmd, cwd=str(cwd), capture_output=True, text=True, env=env, timeout=timeout)
        output=(p.stdout or '')+'\n'+(p.stderr or '')
        (LOGS/f"{label}.log").write_text(output,encoding="utf-8")
        print(output[-2400:] if output.strip() else '(no output)')
        print('Exit code:', p.returncode)
        return p.returncode==0
    except Exception as e:
        (LOGS/f"{label}.log").write_text(repr(e),encoding="utf-8")
        print(type(e).__name__, e)
        return False

print("Python:", sys.executable)
print("Notebook directory:", ROOT)
print("Workspace:", WORK)
print("GPU partition requested? NO")

## 1. 顯示目前的 HPC / Kernel 環境

`CUDA available=False` 屬於正常的 CPU Notebook，不是錯誤。這份 Notebook 不要求 GPU partition。

In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None

def ver(name):
    try: return getattr(importlib.import_module(name),'__version__','installed')
    except Exception: return '(not available)'

rows=[('OS',platform.platform()),('Python',platform.python_version()),('Kernel',sys.executable),
      ('SLURM Job ID',os.getenv('SLURM_JOB_ID','not running under SLURM')),
      ('GPU requested','No'),('PyTorch',ver('torch')),('RDKit',ver('rdkit')),('LMDB',ver('lmdb')),
      ('Git',shutil.which('git') or 'not found')]
display(pd.DataFrame(rows,columns=['Item','Value'])) if pd is not None else print(*rows,sep='\n')
mark('Jupyter CPU kernel','PASS')

## 2. 安裝依賴到**目前 Notebook Kernel**（首次才需要）

使用 `sys.executable -m pip` 保證安裝位置和 Notebook 使用的 Python 相同。**不要在 Notebook 內建立不同的 Conda 環境後仍使用舊 kernel**。若 Python 版本或 HPC 權限不允許安裝，本格會顯示錯誤；後續測試需先修正環境或改用新的 Jupyter kernel。

In [ ]:
# Installed in the active Jupyter kernel; no GPUs, CUDA or SLURM required.
SCIENCE = ['numpy>=1.24,<2','pandas>=1.5,<2.3','scipy<1.14',
           'rdkit>=2023.9,<2025','lmdb>=1.4,<2','matplotlib>=3.6,<4',
           'tqdm<5','pillow','pyyaml<7','iopath<1',
           'tensorboardX<3','scikit-learn>=1.3,<1.6','omegaconf<3',
           'hydra-core<2','packaging<26','ml_collections<2','ipython']
if INSTALL_PACKAGES:
    missing=[pkg for pkg in ['rdkit','lmdb','numpy','pandas','scipy','matplotlib']
             if importlib.util.find_spec(pkg) is None]
    if missing:
        ok=execute([sys.executable,'-m','pip','install',*SCIENCE], 'install_science',timeout=1600)
        mark('Scientific dependencies','PASS' if ok else 'FAIL','restart kernel if required')
    else:
        mark('Scientific dependencies','PASS','already importable')
    if importlib.util.find_spec('torch') is None:
        # 2.2.2 is compatible with Python 3.9–3.11; Python 3.12+ needs newer torch.
        torch_spec = 'torch==2.2.2' if sys.version_info[:2] <= (3,11) else 'torch'
        ok=execute([sys.executable,'-m','pip','install',torch_spec,
                    '--index-url','https://download.pytorch.org/whl/cpu'],
                   'install_torch_cpu',timeout=2400)
        mark('CPU PyTorch','PASS' if ok else 'FAIL')
    else:
        mark('CPU PyTorch','PASS','existing PyTorch will be run on CPU')
else:
    mark('Dependency install','SKIPPED','INSTALL_PACKAGES=False')
print('RDKit import:', ver('rdkit'), '| torch:',ver('torch'),'| lmdb:',ver('lmdb'))

## 3. RDKit — SMILES parsing 與分子圖（不用 GitHub 原始碼）

先用 8 個內建的正確 SMILES 加 1 個故意錯誤的 SMILES。這樣即使 HPC 不允許連 GitHub，也能完成前半段資料測試。

In [ ]:
import numpy as np
import pandas as pd  # Installed in Section 2 if missing
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from IPython.display import display
TEST_SMILES = [
 ('Ethanol','CCO'), ('Benzene','c1ccccc1'), ('Acetic acid','CC(=O)O'),
 ('Pyridine','c1ccncc1'), ('Cyclohexane','C1CCCCC1'),
 ('L-Alanine','C[C@@H](N)C(=O)O'), ('Triethylamine','CCN(CC)CC'),
 ('Lipid-like amine','CCCCCCCCCCCCN(C)C'), ('Broken SMILES','C1CC')]
parse_rows=[]; draw_mols=[]; draw_names=[]
for name, smiles in TEST_SMILES:
    mol=Chem.MolFromSmiles(smiles)
    valid=mol is not None
    parse_rows.append({'Name':name,'SMILES':smiles,'Valid':valid,
                       'Heavy atoms':mol.GetNumHeavyAtoms() if valid else None,
                       'Canonical':Chem.MolToSmiles(mol,isomericSmiles=True) if valid else None})
    if valid:
        draw_mols.append(mol);draw_names.append(name)
parse_df=pd.DataFrame(parse_rows)
display(parse_df)
display(Draw.MolsToGridImage(draw_mols,legends=draw_names,molsPerRow=4,subImgSize=(220,155)))
assert sum(parse_df.Valid)==8 and not parse_df.iloc[-1].Valid
mark('RDKit SMILES parsing','PASS','8 valid; 1 invalid rejected as expected')

## 4. RDKit — 3D conformers / 幾何品質 / 圖表

對每個有效分子用 ETKDGv3 產生 2 個 3D conformers，檢查形狀、有限座標、原子清單與不同構型。這是簡化測試，與 LUMI 論文的正式 conformer 設定不同。**為避免重複 2D views，我們不在 LMDB 加 2D conformer；LUMI 的 `Add2DConformerDataset` 會在 training loader 裡處理。**

In [ ]:
import matplotlib.pyplot as plt
records=[]; geom_rows=[]
for row in parse_df.loc[parse_df.Valid].itertuples():
    mol=Chem.AddHs(Chem.MolFromSmiles(row.SMILES))
    params=AllChem.ETKDGv3(); params.randomSeed=SEED; params.numThreads=1
    ids=list(AllChem.EmbedMultipleConfs(mol,numConfs=2,params=params))
    if len(ids)<2:
        geom_rows.append({'Molecule':row.Name,'Status':'FAIL','3D views':len(ids)}); continue
    if AllChem.MMFFHasAllMoleculeParams(mol):
        AllChem.MMFFOptimizeMoleculeConfs(mol,maxIters=100)
    atoms=[a.GetSymbol() for a in mol.GetAtoms()]
    xyzs=[mol.GetConformer(cid).GetPositions().astype('float32') for cid in ids]
    valid=all(x.shape==(len(atoms),3) and np.isfinite(x).all() for x in xyzs)
    geom_rows.append({'Molecule':row.Name,'Status':'PASS' if valid else 'FAIL',
                      'Atoms':len(atoms),'3D views':len(xyzs),
                      'Mean xyz difference':float(np.linalg.norm(xyzs[0]-xyzs[1],axis=1).mean())})
    if valid:
        records.append({'atoms':atoms,'coordinates':xyzs,
                        'smi':Chem.MolToSmiles(Chem.RemoveHs(mol),isomericSmiles=True)})
display(pd.DataFrame(geom_rows))
mark('RDKit ETKDG conformers','PASS' if len(records)==8 else 'FAIL',f'{len(records)}/8 built')

# Inline 3D plot works even when HPC has no desktop graphical interface
if records:
    ex=records[1]; xyz=ex['coordinates'][0]
    fig=plt.figure(figsize=(6,4)); ax=fig.add_subplot(111,projection='3d')
    atom_col={'C':'#737373','O':'#e85b5b','N':'#5b83da','H':'#dddddd'}
    ax.scatter(xyz[:,0],xyz[:,1],xyz[:,2],
               c=[atom_col.get(atom,'#777777') for atom in ex['atoms']],s=36)
    ax.set(title=f"3D atoms: {ex['smi']}",xlabel='X (Å)',ylabel='Y (Å)',zlabel='Z (Å)')
    plt.show()

## 5. LMDB — Train/valid、資料讀取與品質表格

LMDB 使用 **numeric ASCII keys** `b'0', b'1', ...`，資料為 `{'atoms': ..., 'coordinates': ..., 'smi': ...}`。這裡先不依賴官方 DataLoader，可單獨確認 round trip、座標與 split 沒有重複分子。

In [ ]:
import lmdb, pickle
assert len(records)==8, '請先完成 RDKit 3D 測試'
LMDB_DIR=DATA/'lmdb'
LMDB_DIR.mkdir(parents=True,exist_ok=True)
ordered=sorted(records,key=lambda r:r['smi'])
parts={'train':ordered[:6], 'valid':ordered[6:]}
for split,items in parts.items():
    path=LMDB_DIR/(split+'.lmdb')
    if path.exists(): path.unlink()  # Only overwrite notebook test artifacts
    env=lmdb.open(str(path),map_size=64*1024*1024,subdir=False)
    with env.begin(write=True) as txn:
        for i,item in enumerate(items):
            txn.put(str(i).encode('ascii'),pickle.dumps(item,protocol=4))
    env.close()
roundtrip=[]
for split,items in parts.items():
    env=lmdb.open(str(LMDB_DIR/(split+'.lmdb')),subdir=False,readonly=True,lock=False)
    with env.begin() as txn:
        assert txn.stat()['entries']==len(items)
        for i in range(len(items)):
            rec=pickle.loads(txn.get(str(i).encode('ascii')))
            for xyz in rec['coordinates']:
                assert np.asarray(xyz).shape==(len(rec['atoms']),3)
                assert np.isfinite(xyz).all()
            roundtrip.append({'Split':split,'SMILES':rec['smi'],
                              'Atoms':len(rec['atoms']),'3D views':len(rec['coordinates'])})
    env.close()
check_df=pd.DataFrame(roundtrip)
display(check_df)
assert set(check_df[check_df.Split=='train'].SMILES).isdisjoint(
       set(check_df[check_df.Split=='valid'].SMILES))
mark('LMDB roundtrip & split','PASS','6 train / 2 valid')

## 6. 下載公開研究程式碼：ZIP，不依賴 Git

以下程式會使用 GitHub API 取得 `LUMI-lab`、`Uni-Mol`、`Uni-Core` source ZIP（不是 checkpoint）；如果資料夾已存在，會保留原資料，不會覆蓋。這是對你目前「GitHub 可連線，但 Jupyter 找不到 git」的修正。

> 下載來源只在首次執行時使用；更新研究程式碼時建議另外儲存測試結果與資料，避免覆蓋 notebook 使用中的來源。

In [ ]:
import urllib.request, urllib.error, zipfile, io, tempfile

REPOS = {
    'LUMI-lab': 'bowenli-lab/LUMI-lab',
    'Uni-Mol': 'deepmodeling/Uni-Mol',
    'Uni-Core': 'dptech-corp/Uni-Core',
}

def download_source_zip(repository, destination):
    # GitHub API response is a ZIP archive, NOT pretrained model weights.
    last_error = None
    for branch in ('main', 'master'):
        url = f'https://api.github.com/repos/{repository}/zipball/{branch}'
        try:
            req = urllib.request.Request(url, headers={'User-Agent':'LUMI-HPC-CPU-Validation'})
            with urllib.request.urlopen(req, timeout=120) as r:
                raw = r.read()
            with zipfile.ZipFile(io.BytesIO(raw)) as z:
                files = [info for info in z.infolist() if not info.is_dir()]
                roots = {info.filename.split('/')[0] for info in files}
                if len(roots) != 1: raise ValueError('Unexpected source ZIP layout')
                root = next(iter(roots))
                with tempfile.TemporaryDirectory(dir=str(SOURCE)) as t:
                    target = Path(t)
                    # Safe extraction: do not allow paths outside temporary directory.
                    for info in files:
                        parts = Path(info.filename).parts
                        if not parts or parts[0] != root or '..' in parts: raise ValueError('Unsafe ZIP path')
                        rel = Path(*parts[1:])
                        if not rel.parts: continue
                        out = target / rel
                        out.parent.mkdir(parents=True, exist_ok=True)
                        with z.open(info) as inp, out.open('wb') as dst:
                            shutil.copyfileobj(inp, dst)
                    if destination.exists(): raise FileExistsError(destination)
                    shutil.move(str(target),str(destination))
            return branch
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f'{repository}: {last_error}')

source_rows=[]
for name, repository in REPOS.items():
    dest = SOURCE/name
    if dest.is_dir() and any(dest.iterdir()):
        present=True; note='Reused existing folder'
    elif DOWNLOAD_GITHUB:
        try:
            branch=download_source_zip(repository,dest)
            present=True; note=f'Downloaded ZIP ({branch})'
        except Exception as e:
            present=False; note=f'{type(e).__name__}: {e}'
    else:
        present=False; note='Not found, and downloads disabled'
    source_rows.append({'Repository':name,'Present':present,'Source':repository,'Note':note})
    print(name, ':', 'READY' if present else 'FAILED', note)
mark('GitHub sources','PASS' if all(r['Present'] for r in source_rows) else 'FAIL',
     'ZIP download, no git and no pretrained model weights')
display(pd.DataFrame(source_rows))

## 7. Uni-Core / LUMI data module / 字典相容性檢查

公開 repository 的 layout 可能不同；先列出候選位置，找到後才複製。若缺少自訂 `Add2DConformerDataset`，會讓之後的 imports 明確失敗，而非聲稱已補齊。安裝 Uni-Core 可能需要與 HPC Python 版本相容的依賴和 C/C++ toolchain。

In [ ]:
CORE=SOURCE/'Uni-Core'
if (CORE/'setup.py').exists():
    try:
        installed = importlib.util.find_spec('unicore') is not None
    except Exception:
        installed = False
    if not installed and INSTALL_PACKAGES:
        ok=execute([sys.executable,'-m','pip','install','--no-build-isolation','--no-deps',
                    '-e',str(CORE)],'install_unicore',timeout=1200)
        importlib.invalidate_caches()
    else:
        ok=installed
    mark('Uni-Core install attempt','PASS' if ok else 'FAIL',
         'PASS means installer completed; imports checked separately')
else:
    mark('Uni-Core install attempt','FAIL','Source/setup.py unavailable')

upstream=SOURCE/'Uni-Mol'
data_candidates=[
    upstream/'unimol'/'unimol'/'data',
    upstream/'unimol'/'data',
]
# If the layout changes, inspect other candidate folders without assuming any is compatible.
data_candidates += [p for p in upstream.rglob('data') if p.is_dir() and (p/'__init__.py').exists()]
UP=next((p for p in data_candidates if (p/'__init__.py').is_file() and
         ('LMDBDataset' in (p/'__init__.py').read_text(errors='ignore'))), None)
LUMI_DATA=SOURCE/'LUMI-lab'/'model'/'unimol'/'data'
if UP is not None and (SOURCE/'LUMI-lab'/'model'/'unimol').is_dir():
    if not LUMI_DATA.exists():
        shutil.copytree(UP,LUMI_DATA)
    print('Upstream data source:', UP)
else:
    print('No matching upstream Uni-Mol data module found. Candidates:',data_candidates[:8])

# Search actual dictionary location (do not assume a hard-coded path).
dicts=list(upstream.rglob('dict.txt')) if upstream.is_dir() else []
print('Dictionary candidates:',[str(p.relative_to(upstream)) for p in dicts[:10]])
if dicts:
    # Prefer Uni-Mol example molecule dictionary if present.
    molecule_dicts=[p for p in dicts if 'molecule' in str(p).lower()]
    selected=(molecule_dicts or dicts)[0]
    shutil.copy2(selected,LMDB_DIR/'dict.txt')
    print('Selected dictionary:',selected)
else:
    print('WARNING: no atom dictionary found')
mark('LUMI data module + dictionary',
     'PASS' if LUMI_DATA.is_dir() and (LMDB_DIR/'dict.txt').is_file() else 'FAIL',
     'Presence only; LUMI import test checks compatibility')
(OUTPUT/'source_provenance.json').write_text(json.dumps(source_rows,indent=2,ensure_ascii=False))

## 8. 真正的 LUMI import 相容性報告

不使用 `unimol_tools` 代替 LUMI。如果原始碼或依賴不完整，顯示 FAIL，之後的 LUMI model tests 就會 SKIP；前面的 RDKit 和 LMDB 表格仍完整保存。

In [ ]:
LUMI_MODEL_DIR=SOURCE/'LUMI-lab'/'model'
if LUMI_MODEL_DIR.is_dir() and str(LUMI_MODEL_DIR) not in sys.path:
    sys.path.insert(0,str(LUMI_MODEL_DIR))
needed=['torch','unicore','unimol','unimol.data',
        'unimol.tasks.unimol_contrastive',
        'unimol.models.unimol_contrastive',
        'unimol.losses.unimol_contrastive']
imp=[]
for modname in needed:
    try:
        mod=importlib.import_module(modname)
        imp.append({'Module':modname,'Status':'PASS','Error':''})
    except Exception as e:
        imp.append({'Module':modname,'Status':'FAIL','Error':f'{type(e).__name__}: {e}'[:220]})
imp_df=pd.DataFrame(imp); display(imp_df)
LUMI_READY=all(x['Status']=='PASS' for x in imp) and (LMDB_DIR/'dict.txt').is_file()
mark('LUMI dependency compatibility','PASS' if LUMI_READY else 'FAIL',
     'if failed, fix dependencies before training')

## 9. LUMI Contrastive Dataset Loader（CPU）

真實使用 LUMI 的 `UniMolTaskContrastive`，從 LMDB 讀取兩個分子的兩個 views，並列出 input tensor 形狀及正樣本配對。這是 **真正的 LUMI DataLoader**，不是 mock。

In [ ]:
import torch
from types import SimpleNamespace
TASK=None; BATCH=None
if not LUMI_READY:
    mark('LUMI paired-view DataLoader','SKIPPED','dependency imports not ready')
else:
    try:
        from unimol.tasks.unimol_contrastive import UniMolTaskContrastive
        args=SimpleNamespace(data=str(LMDB_DIR),dict_name='dict.txt',seed=SEED,
            only_polar=0,remove_hydrogen=False,remove_polar_hydrogen=False,
            mode='train',max_atoms=256,max_seq_len=256,
            mask_prob=0.4,leave_unmasked_prob=0.0,random_token_prob=0.0,
            noise_type='uniform',noise=0.5,contrastive_loss=1.0)
        TASK=UniMolTaskContrastive.setup_task(args)
        TASK.load_dataset('train')
        d=TASK.dataset('train')
        BATCH=d.collater([d[0],d[1]])
        names=list(BATCH['target']['smi_name'])
        assert BATCH['net_input']['src_tokens'].shape[0]==4
        assert names[:2]==names[2:],f'view pairing order unexpected: {names}'
        display(pd.DataFrame([
          {'Tensor':key,'Shape':tuple(val.shape)}
          for key,val in BATCH['net_input'].items() if isinstance(val,torch.Tensor)]))
        display(pd.DataFrame({'View1':names[:2],'View2':names[2:]}))
        mask_count=int(BATCH['target']['tokens_target'].ne(TASK.dictionary.pad()).sum())
        assert mask_count>0
        mark('LUMI paired-view DataLoader','PASS',f'2 molecules / 4 views; masked atoms {mask_count}')
    except Exception as e:
        (LOGS/'loader_error.txt').write_text(traceback.format_exc())
        print(traceback.format_exc(limit=7))
        mark('LUMI paired-view DataLoader','FAIL',str(e))
        TASK=None; BATCH=None

## 10. LUMI 小型 Transformer — **CPU 隨機初始化** Forward + Backward

我們只測試工程功能：**1 layer、64 hidden dimension、4 heads**，不載入任何 pretrained weights。正式 LUMI model 比這個測試設定大很多。這裡測試原始自訂 model + loss，而不是替代的玩具模型。

In [ ]:
MODEL=None; CRITERION=None; OPTIMIZER=None
if not RUN_MODEL_TESTS or TASK is None or BATCH is None:
    mark('LUMI CPU forward/backward','SKIPPED','disabled or loader unavailable')
else:
    try:
        from unimol.models.unimol_contrastive import UniMolModelContrastive
        from unimol.losses.unimol_contrastive import UniMolLossContrastive
        torch.set_num_threads(min(2,os.cpu_count() or 1)); torch.manual_seed(SEED)
        cfg=TASK.args
        cfg.encoder_layers=1; cfg.encoder_embed_dim=64
        cfg.encoder_ffn_embed_dim=128; cfg.encoder_attention_heads=4
        cfg.masked_token_loss=1.0; cfg.masked_coord_loss=5.0
        cfg.masked_dist_loss=10.0; cfg.contrastive_loss=1.0
        cfg.x_norm_loss=0.01; cfg.delta_pair_repr_norm_loss=0.01
        cfg.masked_coord_dist_loss=0.0;cfg.mode='train'
        MODEL=UniMolModelContrastive(cfg,TASK.dictionary).cpu().train()
        CRITERION=UniMolLossContrastive(TASK)
        OPTIMIZER=torch.optim.Adam(MODEL.parameters(),lr=1e-4)
        OPTIMIZER.zero_grad(set_to_none=True)
        loss,_,logs=CRITERION(MODEL,BATCH)
        assert torch.isfinite(loss).item()
        loss.backward()
        ngrads=sum(p.grad is not None and torch.isfinite(p.grad).all().item()
                   for p in MODEL.parameters())
        assert ngrads>0, 'No finite gradients'
        OPTIMIZER.step()
        metrics=[]
        for key,value in logs.items():
            if isinstance(value,(int,float)) or (torch.is_tensor(value) and value.numel()==1):
                metrics.append({'Metric':key,'Value':float(value)})
        display(pd.DataFrame(metrics))
        print('Parameters:',sum(p.numel() for p in MODEL.parameters()),
              'Parameters with finite gradient:',ngrads)
        mark('LUMI CPU forward/backward','PASS','genuine model, random initialization')
    except Exception as e:
        print(traceback.format_exc(limit=8))
        (LOGS/'forward_error.txt').write_text(traceback.format_exc())
        mark('LUMI CPU forward/backward','FAIL',str(e))
        MODEL=None

## 11. Mini-pretraining、Loss 曲線與 Checkpoint Roundtrip

這只是 **3 個 CPU optimizer updates 的 smoke test**，無法證明模型已學好化學結構。訓練過程所有 loss 直接顯示於 Notebook。

In [ ]:
if MODEL is None or CRITERION is None or OPTIMIZER is None:
    mark('CPU mini-training + checkpoint','SKIPPED','forward/backward not ready')
else:
    try:
        start=time.time(); rows=[]
        for step in range(1,MINI_TRAIN_STEPS+1):
            OPTIMIZER.zero_grad(set_to_none=True)
            loss,_,log=CRITERION(MODEL,BATCH)
            assert torch.isfinite(loss).item()
            loss.backward();OPTIMIZER.step()
            rec={'Step':step,'Total Loss':float(loss.detach())}
            for field in ['contrastive_loss','masked_token_loss','masked_coord_loss','masked_dist_loss']:
                value=log.get(field, np.nan)
                rec[field]=float(value) if (torch.is_tensor(value) and value.numel()==1) or isinstance(value,(int,float)) else np.nan
            rows.append(rec)
            print(f"step {step}/{MINI_TRAIN_STEPS}, loss {rec['Total Loss']:.5f}")
        results=pd.DataFrame(rows);display(results)
        ax=results.plot(x='Step',y=['Total Loss','contrastive_loss','masked_token_loss'],
                        figsize=(7,4),marker='o')
        ax.set(title='LUMI random-init CPU smoke test (NOT model convergence)',ylabel='Loss')
        plt.show()
        path=OUTPUT/'scratch_cpu_checkpoint.pt'
        torch.save({'model':MODEL.state_dict(),'optimizer':OPTIMIZER.state_dict(),
                    'completed_steps':MINI_TRAIN_STEPS},path)
        try:
            loaded=torch.load(path,map_location='cpu',weights_only=True)
        except TypeError:
            loaded=torch.load(path,map_location='cpu')
        MODEL.load_state_dict(loaded['model'],strict=True)
        assert loaded['completed_steps']==MINI_TRAIN_STEPS
        results.to_csv(OUTPUT/'loss_history.csv',index=False)
        mark('CPU mini-training + checkpoint','PASS',f'{MINI_TRAIN_STEPS} steps, {time.time()-start:.1f}s; {path.name}')
    except Exception as e:
        print(traceback.format_exc(limit=8))
        (LOGS/'training_error.txt').write_text(traceback.format_exc())
        mark('CPU mini-training + checkpoint','FAIL',str(e))

## 12. PASS / FAIL Dashboard + 可帶回此對話的結果

**請貼回 Dashboard 表格與第一個 FAIL 的 trace**，這比只說「Jupyter 有錯」更容易定位。未使用 GPU、無 SLURM partition。`status.json` 方便搬移。

In [ ]:
summary=pd.DataFrame(STATUS.values())
display(summary)
print('PASS:',sum(summary.status=='PASS'),'FAIL:',sum(summary.status=='FAIL'),
      'SKIPPED:',sum(summary.status=='SKIPPED'))
(OUTPUT/'status.json').write_text(json.dumps(list(STATUS.values()),indent=2,ensure_ascii=False),encoding='utf8')
print('Results:',OUTPUT)
print('Log files:',LOGS)

## 如何解讀結果？

- **RDKit/LMDB PASS，但 LUMI imports FAIL：** 分子資料沒有問題；需要處理 Uni-Core / Uni-Mol fork 的版本或 `unimol.data` 相容性。**不是 GPU partition 的錯。**
- **DataLoader PASS、Forward FAIL：** 請提供 `logs/forward_error.txt`，檢查 loss/model 的 API 版本。
- **Forward PASS，Checkpoint PASS：** CPU 已證明 **LUMI 的小型隨機初始化訓練路徑可以運作**；但要進行真正 general molecular pretraining，還需要較大的資料集、化學結構品質管控、訓練策略及正式 GPU 配置。
- **Notebook 在執行前就卡在 JupyterHub「GPU partition」：** 這是 Jupyter server 啟動時所選的 SLURM profile 問題。Notebook 內的 Python 指令無法修改尚未成功啟動的 server。請使用 CPU/Jupyter profile。

來源：[LUMI-lab](https://github.com/bowenli-lab/LUMI-lab) · [官方 Uni-Mol](https://github.com/deepmodeling/Uni-Mol) · [Uni-Core](https://github.com/dptech-corp/Uni-Core)